# Sensitivity Analysis Figures

Workflow: edit the function → run the show cell to preview → run the save cell.

In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd
import yaml

In [ ]:
def find_project_root():
    path = Path.cwd()
    for _ in range(5):
        if (path / "code").is_dir() and (path / "results").is_dir():
            return path
        path = path.parent
    return Path.cwd()


PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "code" / "settings" / "sensitivity_config.yaml"
RESULTS_DIR = PROJECT_ROOT / "results" / "sensitivity"
FIGURES_DIR = RESULTS_DIR / "figs"
FIGURES_DIR.mkdir(exist_ok=True)

print(f"Project : {PROJECT_ROOT}")
print(f"Config  : {CONFIG_PATH}")
print(f"Output  : {RESULTS_DIR}")

In [ ]:
# ── parameter categories (hard-coded) ─────────────────────────────────────

PARAM_CATEGORIES = {
    "n_households": "Structural",
    "search_radius": "Structural",
    "risk_aversion_mu": "Behavioural",
    "loss_aversion": "Behavioural",
    "dti_limit": "Credit",
    "mortgage_rate": "Credit",
    "btl_funding_rate": "Credit",
    "inst_funding_rate": "Credit",
}

CATEGORY_ORDER = ["Structural", "Behavioural", "Credit"]

CATEGORY_COLORS = {
    "Structural": "#4c72b0",
    "Behavioural": "#dd8452",
    "Credit": "#55a868",
}

In [ ]:
# ── labels (edit these to change titles / axis labels) ────────────────────

PARAM_LABELS = {
    "n_households": "N households",
    "search_radius": "Search radius",
    "risk_aversion_mu": "Risk aversion",
    "loss_aversion": "Loss aversion",
    "dti_limit": "DTI limit",
    "mortgage_rate": "Mortgage rate",
    "btl_funding_rate": "BTL funding rate",
    "inst_funding_rate": "Institutional funding rate",
}

RESPONSE_LABELS = {
    "mean_price": "Mean Sale Price",
    "price_volatility": "Price Volatility",
    "oo_ownership_share": "Owner-Occupier Share",
    "ll_ownership_share": "Landlord Share",
    "inst_ownership_share": "Institutional Share",
    "avg_net_worth": "Avgerage Household Net Worth",
    "net_worth_gini": "Wealth Gini",
}

# Response groups for the themed bar charts
RESPONSE_GROUPS = {
    "Ownership Distribution": [
        "oo_ownership_share",
        "ll_ownership_share",
        "inst_ownership_share",
    ],
    "Market Prices": [
        "mean_price",
        "price_volatility",
    ],
    "Household Welfare": [
        "avg_net_worth",
        "net_worth_gini",
    ],
}

In [ ]:
# ── helpers ───────────────────────────────────────────────────────────────


def load_config():
    with open(CONFIG_PATH) as f:
        return yaml.safe_load(f)


def load_sobol():
    path = RESULTS_DIR / "sobol_indices.csv"
    if not path.exists():
        raise FileNotFoundError(f"{path} not found")
    return pd.read_csv(path)


def order_params_by_category(param_names):
    ordered = []
    for cat in CATEGORY_ORDER:
        for p in param_names:
            if PARAM_CATEGORIES.get(p) == cat:
                ordered.append(p)
    for p in param_names:
        if p not in ordered:
            ordered.append(p)
    return ordered


def param_tick_labels(param_order):
    return [PARAM_LABELS.get(p, p) for p in param_order]

In [ ]:
sa_cfg = load_config()
sobol_df = load_sobol()
param_names = [p["name"] for p in sa_cfg["parameters"]]

print(f"Parameters ({len(param_names)}): {param_names}")
for group, resp_names in RESPONSE_GROUPS.items():
    print(f"  {group}: {resp_names}")

---
## Figure 1: Sobol Heatmap

### 1a. Function

In [ ]:
def plot_sobol_heatmap(
    sobol_df,
    param_names,
    resp_names,
    resp_labels,
    vmax=None,
    annotate_threshold=0.02,
    figsize=(14, 7),
    cmap="YlOrRd",
):
    param_order = order_params_by_category(param_names)
    n_params = len(param_order)
    n_resp = len(resp_names)

    s1_mat = np.full((n_resp, n_params), np.nan)
    st_mat = np.full((n_resp, n_params), np.nan)
    for i, resp in enumerate(resp_names):
        sub = sobol_df[sobol_df["response"] == resp]
        for j, param in enumerate(param_order):
            row = sub[sub["parameter"] == param]
            if not row.empty:
                s1_mat[i, j] = row["S1"].values[0]
                st_mat[i, j] = row["ST"].values[0]

    if vmax is None:
        vmax = max(np.nanmax(s1_mat), np.nanmax(st_mat))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)

    cat_strip = np.full((1, n_params), np.nan)
    seen_cats = {}
    for j, p in enumerate(param_order):
        cat = PARAM_CATEGORIES.get(p, "")
        if cat not in seen_cats:
            seen_cats[cat] = len(seen_cats)
        cat_strip[0, j] = seen_cats[cat]
    cat_cmap = mpl.colors.ListedColormap([CATEGORY_COLORS[c] for c in CATEGORY_ORDER])
    cat_bounds = np.arange(len(CATEGORY_ORDER) + 1) - 0.5
    cat_norm = mpl.colors.BoundaryNorm(cat_bounds, len(CATEGORY_ORDER))

    for ax, mat, title in zip(
        [ax1, ax2], [s1_mat, st_mat], ["First-order (S₁)", "Total-order (Sₜ)"]
    ):
        full_mat = np.vstack([cat_strip, mat])
        im = ax.imshow(full_mat, cmap=cmap, aspect="auto", vmin=0, vmax=vmax)
        ax.imshow(
            cat_strip,
            cmap=cat_cmap,
            norm=cat_norm,
            aspect="auto",
            extent=[-0.5, n_params - 0.5, -0.5, 0.5],
        )

        for cat in CATEGORY_ORDER:
            cols = [j for j, p in enumerate(param_order) if PARAM_CATEGORIES.get(p) == cat]
            if cols:
                mid = (cols[0] + cols[-1]) / 2
                ax.text(
                    mid,
                    0,
                    cat,
                    ha="center",
                    va="center",
                    fontsize=8,
                    fontweight="bold",
                    color="white",
                )

        for i in range(n_resp):
            for j in range(n_params):
                val = mat[i, j]
                if np.isnan(val):
                    continue
                c = "white" if val > vmax * 0.55 else "#333"
                txt = f"{val:.2f}" if abs(val) >= annotate_threshold else ""
                ax.text(j, i + 1, txt, ha="center", va="center", fontsize=7, color=c)

        tick_labels = param_tick_labels(param_order)
        ax.set_xticks(range(n_params))
        ax.set_xticklabels(tick_labels, rotation=35, ha="right", fontsize=9)
        ax.set_yticks(range(n_resp + 1))
        ax.set_yticklabels([""] + resp_labels, fontsize=9)
        ax.set_title(title, fontweight="bold", fontsize=12)

    fig.colorbar(im, ax=[ax1, ax2], shrink=0.6, pad=0.02, label="Sobol index")
    fig.subplots_adjust(wspace=0.3)
    return fig

### 1b. Show

In [ ]:
all_resp_names = sum(RESPONSE_GROUPS.values(), [])
all_resp_labels = [RESPONSE_LABELS[r] for r in all_resp_names]

fig1 = plot_sobol_heatmap(sobol_df, param_names, all_resp_names, all_resp_labels)
plt.show()

### 1c. Save

In [ ]:
fig1.savefig(FIGURES_DIR / "sobol_heatmap.png", dpi=300, bbox_inches="tight")
plt.close(fig1)
print("Saved: sobol_heatmap.png")

---
## Figure 2: S1 vs ST Scatter

### 2a. Function

In [ ]:
def plot_s1_vs_st_scatter(sobol_df, param_names, figsize=(8, 7), label_extreme=True):
    fig, ax = plt.subplots(figsize=figsize)

    for param in param_names:
        sub = sobol_df[sobol_df["parameter"] == param]
        if sub.empty:
            continue
        cat = PARAM_CATEGORIES.get(param, "Other")
        color = CATEGORY_COLORS.get(cat, "#999")
        ax.scatter(
            sub["S1"],
            sub["ST"],
            label=PARAM_LABELS.get(param, param),
            color=color,
            alpha=0.75,
            edgecolors="white",
            linewidth=0.5,
            s=70,
            zorder=5,
        )

    lims = [
        min(ax.get_xlim()[0], ax.get_ylim()[0]) - 0.02,
        max(ax.get_xlim()[1], ax.get_ylim()[1]) + 0.02,
    ]
    ax.plot(lims, lims, "k--", alpha=0.35, linewidth=1, label="S₁ = Sₜ", zorder=1)
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_xlabel("First-order index (S₁)", fontsize=11)
    ax.set_ylabel("Total-order index (Sₜ)", fontsize=11)
    ax.set_title("Sobol indices: S₁ vs Sₜ", fontweight="bold", fontsize=13)
    ax.grid(alpha=0.25)
    ax.legend(
        bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8, framealpha=0.9, edgecolor="#ccc"
    )

    if label_extreme:
        for _, row in sobol_df.iterrows():
            gap = row["ST"] - row["S1"]
            if gap > 0.25:
                param_label = PARAM_LABELS.get(row["parameter"], row["parameter"])
                resp_label = RESPONSE_LABELS.get(row["response"], row["response"][:12])
                ax.annotate(
                    f"{param_label} / {resp_label}",
                    (row["S1"], row["ST"]),
                    xytext=(5, 5),
                    textcoords="offset points",
                    fontsize=6,
                    alpha=0.7,
                    arrowprops=dict(arrowstyle="->", color="grey", alpha=0.5),
                )

    fig.tight_layout()
    return fig

### 2b. Show

In [ ]:
fig2 = plot_s1_vs_st_scatter(sobol_df, param_names)
plt.show()

### 2c. Save

In [ ]:
fig2.savefig(FIGURES_DIR / "sobol_s1_vs_st.png", dpi=300, bbox_inches="tight")
plt.close(fig2)
print("Saved: sobol_s1_vs_st.png")

---
## Figures 3A–C: Thematic Bar Charts

A single function that plots any subset of responses as a row.

### Function (shared)

In [ ]:
def plot_thematic_bars(sobol_df, param_names, resp_names, resp_labels, fig_title="", figsize=None):
    from matplotlib.patches import Patch

    param_order = order_params_by_category(param_names)
    n_params = len(param_order)
    n_resp = len(resp_names)
    n_cols = n_resp
    n_rows = 1

    if figsize is None:
        figsize = (4.5 * n_cols, 4.5)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize, squeeze=False)
    fig.suptitle(fig_title, fontsize=14, fontweight="bold", y=1.03)

    x = np.arange(n_params)
    width = 0.35
    tick_labels = param_tick_labels(param_order)

    for ax, resp_name, resp_label in zip(axes.flat, resp_names, resp_labels):
        sub = sobol_df[sobol_df["response"] == resp_name]
        if sub.empty:
            ax.text(
                0.5,
                0.5,
                "N/A",
                ha="center",
                va="center",
                transform=ax.transAxes,
                fontsize=12,
                color="#999",
            )
            ax.set_title(resp_label, fontsize=11)
            continue

        s1_vals, s1_err, st_vals, st_err, bar_colors = [], [], [], [], []
        for p in param_order:
            row = sub[sub["parameter"] == p]
            if row.empty:
                s1_vals.append(0)
                s1_err.append(0)
                st_vals.append(0)
                st_err.append(0)
            else:
                s1_vals.append(row["S1"].values[0])
                s1_err.append(row["S1_conf"].values[0])
                st_vals.append(row["ST"].values[0])
                st_err.append(row["ST_conf"].values[0])
            bar_colors.append(CATEGORY_COLORS.get(PARAM_CATEGORIES.get(p, ""), "#ccc"))

        ax.bar(
            x - width / 2,
            s1_vals,
            width,
            yerr=s1_err,
            color=bar_colors,
            capsize=3,
            label="1st order",
            edgecolor="white",
            linewidth=0.3,
        )
        ax.bar(
            x + width / 2,
            st_vals,
            width,
            yerr=st_err,
            color=bar_colors,
            capsize=3,
            label="Total order",
            edgecolor="white",
            linewidth=0.3,
            alpha=0.55,
        )

        ax.axhline(0, color="grey", linewidth=0.5)
        ax.set_xticks(x)
        ax.set_xticklabels(tick_labels, rotation=30, ha="right", fontsize=8)
        ax.set_title(resp_label, fontsize=11)
        ax.grid(axis="y", alpha=0.25)

        seen = set()
        for j, p in enumerate(param_order):
            cat = PARAM_CATEGORIES.get(p, "")
            if cat and cat not in seen:
                seen.add(cat)
                if j > 0:
                    ax.axvline(
                        j - 0.5, color="grey", linewidth=0.6, linestyle=":", alpha=0.5, zorder=0
                    )

    cat_patches = [
        Patch(facecolor=c, label=cat, edgecolor="white") for cat, c in CATEGORY_COLORS.items()
    ]
    order_patches = [
        Patch(facecolor="#555", label="1st order"),
        Patch(facecolor="#555", alpha=0.55, label="Total order"),
    ]

    legend1 = fig.legend(
        handles=cat_patches,
        loc="lower center",
        ncol=3,
        fontsize=9,
        framealpha=0.9,
        bbox_to_anchor=(0.35, -0.06),
    )
    fig.legend(
        handles=order_patches,
        loc="lower center",
        ncol=2,
        fontsize=9,
        framealpha=0.9,
        bbox_to_anchor=(0.73, -0.06),
    )
    fig.add_artist(legend1)
    fig.tight_layout()
    return fig

### 3A. Ownership Distribution

In [ ]:
resp_names_3a = RESPONSE_GROUPS["Ownership Distribution"]
resp_labels_3a = [RESPONSE_LABELS[r] for r in resp_names_3a]

fig3a = plot_thematic_bars(
    sobol_df, param_names, resp_names_3a, resp_labels_3a, fig_title="Ownership Distribution"
)
plt.show()

In [ ]:
fig3a.savefig(FIGURES_DIR / "sobol_ownership_structure.png", dpi=300, bbox_inches="tight")
plt.close(fig3a)
print("Saved: sobol_ownership_structure.png")

### 3B. Market Prices

In [ ]:
resp_names_3b = RESPONSE_GROUPS["Market Prices"]
resp_labels_3b = [RESPONSE_LABELS[r] for r in resp_names_3b]

fig3b = plot_thematic_bars(
    sobol_df, param_names, resp_names_3b, resp_labels_3b, fig_title="Market Prices"
)
plt.show()

In [ ]:
fig3b.savefig(FIGURES_DIR / "sobol_market_prices.png", dpi=300, bbox_inches="tight")
plt.close(fig3b)
print("Saved: sobol_market_prices.png")

### 3C. Household Welfare

In [ ]:
resp_names_3c = RESPONSE_GROUPS["Household Welfare"]
resp_labels_3c = [RESPONSE_LABELS[r] for r in resp_names_3c]

fig3c = plot_thematic_bars(
    sobol_df, param_names, resp_names_3c, resp_labels_3c, fig_title="Household Welfare"
)
plt.show()

In [ ]:
fig3c.savefig(FIGURES_DIR / "sobol_household_welfare.png", dpi=300, bbox_inches="tight")
plt.close(fig3c)
print("Saved: sobol_household_welfare.png")